# Aggergated Tests

This notebook runs all test suites across the project and produces a summary + data & graphs which are useful to evaluate the state of the project.

## Modules Tested

The modules tested are:
* ML pipeline - windowing, features, scaler, LOSO, TFlite parity, FL sim;
* Server API - auth, apis, k-anon, audit, privacy;
* mobile - FeatureExtractor, ScalerNormalizer, XAI grouping;
* wear - Buffer flushing and schemas.


In [78]:
import subprocess
import sys
import os
from pathlib import Path
from IPython.display import display, Markdown

REPO_ROOT = Path(os.getcwd())
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent.parent
elif REPO_ROOT.name == 'ml':
    REPO_ROOT = REPO_ROOT.parent

assert (REPO_ROOT / 'ml').exists() and (REPO_ROOT / 'server').exists(), \
    f"Cannot locate repo root, got: {REPO_ROOT}"

PYTHON = sys.executable
print(f"Repository root: {REPO_ROOT}")
print(f"Python:          {PYTHON}")


Repository root: c:\Users\daniion6\AndroidStudioProjects\MindWave
Python:          c:\Users\daniion6\AndroidStudioProjects\MindWave\.venv-312\Scripts\python.exe


## ML Pipeline Tests

In [79]:
ml_result = subprocess.run(
    [PYTHON, "-m", "pytest", "tests/", "-v", "--tb=short", "--no-header"],
    cwd=str(REPO_ROOT / "ml"),
    capture_output=True, text=True
)
print(ml_result.stdout)
if ml_result.returncode != 0:
    print("STDERR:", ml_result.stderr)


============================= test session starts =============================
collecting ... collected 76 items

tests/test_features.py::TestFeatureNameConsistency::test_n_features_is_23 PASSED [  1%]
tests/test_features.py::TestFeatureNameConsistency::test_feature_name_count PASSED [  2%]
tests/test_features.py::TestFeatureNameConsistency::test_feature_group_counts PASSED [  3%]
tests/test_features.py::TestFeatureNameConsistency::test_concatenation_order PASSED [  5%]
tests/test_features.py::TestHRVTimeFeatures::test_output_length PASSED   [  6%]
tests/test_features.py::TestHRVTimeFeatures::test_short_signal_returns_zeros PASSED [  7%]
tests/test_features.py::TestHRVTimeFeatures::test_constant_signal_returns_zeros PASSED [  9%]
tests/test_features.py::TestHRVFreqFeatures::test_output_length PASSED   [ 10%]
tests/test_features.py::TestHRVFreqFeatures::test_no_nan_inf PASSED      [ 11%]
tests/test_features.py::TestHRVFreqFeatures::test_short_signal_zeros PASSED [ 13%]
tests/test_featu

## Server API Tests

In [80]:
server_result = subprocess.run(
    [PYTHON, "-m", "pytest", "tests/", "-v", "--tb=short", "--no-header"],
    cwd=str(REPO_ROOT / "server"),
    capture_output=True, text=True
)
print(server_result.stdout)
if server_result.returncode != 0:
    print("STDERR:", server_result.stderr)

============================= test session starts =============================
collecting ... collected 40 items

tests/test_audit.py::TestAuditLogging::test_post_creates_audit_entry PASSED [  2%]
tests/test_audit.py::TestAuditLogging::test_get_does_not_create_audit_entry PASSED [  5%]
tests/test_audit.py::TestAuditLogging::test_audit_captures_user_id PASSED [  7%]
tests/test_audit.py::TestAuditLogging::test_unauthenticated_post_still_logged PASSED [ 10%]
tests/test_auth.py::TestRegister::test_public_register_success PASSED    [ 12%]
tests/test_auth.py::TestRegister::test_duplicate_email_rejected PASSED   [ 15%]
tests/test_auth.py::TestRegister::test_weak_password_rejected PASSED     [ 17%]
tests/test_auth.py::TestRegister::test_password_requires_digit PASSED    [ 20%]
tests/test_auth.py::TestRegister::test_admin_register_requires_auth PASSED [ 22%]
tests/test_auth.py::TestLogin::test_login_success PASSED                 [ 25%]
tests/test_auth.py::TestLogin::test_login_wrong_password 

## Mobile & Wear Unit Tests

Run via Gradle. Requires Android SDK.

In [81]:
import shutil, re

gradle_cmd = str(REPO_ROOT / "gradlew.bat") if os.name == "nt" else str(REPO_ROOT / "gradlew")

def _resolve_java() -> str | None:
    # get java path
    gp = REPO_ROOT / "gradle.properties"
    if gp.exists():
        for line in gp.read_text().splitlines():
            if line.startswith("org.gradle.java.home"):
                jh = line.split("=", 1)[1].strip().replace("\\\\", "\\")
                candidate = Path(jh) / "bin" / "java.exe"
                if candidate.exists():
                    return str(candidate)
    # java home env var
    jh = os.environ.get("JAVA_HOME", "")
    if jh:
        candidate = Path(jh) / "bin" / "java.exe"
        if candidate.exists():
            return str(candidate)
        
    # Android Studio bundled JBR (standard Windows install path)
    as_jbr = Path(r"C:\Program Files\Android\Android Studio\jbr\bin\java.exe")
    if as_jbr.exists():
        return str(as_jbr)
    return shutil.which("java")

def _java_ok() -> tuple[bool, str]:
    # Requires 64-bit JDK 17+
    java = _resolve_java()
    if not java:
        return False, "java not found - install JDK 17+ or set org.gradle.java.home in gradle.properties"
    if r"Program Files (x86)" in java:
        return False, f"32-bit JVM detected ({java}). Gradle requires 64-bit JDK 17+."
    r = subprocess.run([java, "-version"], capture_output=True, text=True)
    ver_line = r.stderr or r.stdout
    m = re.search(r'version "(\d+)', ver_line)
    major = int(m.group(1)) if m else 0
    if major < 17:
        return False, f"JDK {major} detected; required JDK 17+."
    return True, f"JDK {major} at {java}"

java_ok, java_msg = _java_ok()
if not java_ok:
    print(f"Skipping Gradle mobile tests: {java_msg}")
    mobile_result = subprocess.CompletedProcess(args=[], returncode=1, stdout="", stderr=java_msg)
else:
    print(f"Java OK: {java_msg}")
    mobile_result = subprocess.run(
        [gradle_cmd, ":mobile:testDebugUnitTest", "--no-daemon"],
        cwd=str(REPO_ROOT),
        capture_output=True, text=True
    )
    print(mobile_result.stdout[-3000:] if len(mobile_result.stdout) > 3000 else mobile_result.stdout)
    if mobile_result.returncode != 0:
        print("STDERR (last 2000):", mobile_result.stderr[-2000:])


Java OK: JDK 21 at C:\Program Files\Android\Android Studio\jbr\bin\java.exe
To honour the JVM settings for this build a single-use Daemon process will be forked. For more on this, please refer to https://docs.gradle.org/8.13/userguide/gradle_daemon.html#sec:disabling_the_daemon in the Gradle documentation.
Daemon will be stopped at the end of the build 
> Task :mobile:checkKotlinGradlePluginConfigurationErrors SKIPPED
> Task :mobile:preBuild UP-TO-DATE
> Task :mobile:preDebugBuild UP-TO-DATE
> Task :mobile:checkDebugAarMetadata UP-TO-DATE
> Task :mobile:processDebugNavigationResources UP-TO-DATE
> Task :mobile:compileDebugNavigationResources UP-TO-DATE
> Task :mobile:generateDebugResValues UP-TO-DATE
> Task :mobile:mapDebugSourceSetPaths UP-TO-DATE
> Task :mobile:generateDebugResources UP-TO-DATE
> Task :mobile:mergeDebugResources UP-TO-DATE
> Task :mobile:packageDebugResources UP-TO-DATE
> Task :mobile:parseDebugLocalResources UP-TO-DATE
> Task :mobile:createDebugCompatibleScreenManif

In [82]:
if not java_ok:
    print(f"Skipping Gradle wear tests: {java_msg}")
    wear_result = subprocess.CompletedProcess(args=[], returncode=1, stdout="", stderr=java_msg)
else:
    wear_result = subprocess.run(
        [gradle_cmd, ":wear:testDebugUnitTest", "--no-daemon"],
        cwd=str(REPO_ROOT),
        capture_output=True, text=True
    )
    print(wear_result.stdout[-3000:] if len(wear_result.stdout) > 3000 else wear_result.stdout)
    if wear_result.returncode != 0:
        print("STDERR (last 2000):", wear_result.stderr[-2000:])


To honour the JVM settings for this build a single-use Daemon process will be forked. For more on this, please refer to https://docs.gradle.org/8.13/userguide/gradle_daemon.html#sec:disabling_the_daemon in the Gradle documentation.
Daemon will be stopped at the end of the build 
> Task :wear:checkKotlinGradlePluginConfigurationErrors SKIPPED
> Task :wear:preBuild UP-TO-DATE
> Task :wear:preDebugBuild UP-TO-DATE
> Task :wear:checkDebugAarMetadata UP-TO-DATE
> Task :wear:processDebugNavigationResources UP-TO-DATE
> Task :wear:compileDebugNavigationResources UP-TO-DATE
> Task :wear:generateDebugResValues UP-TO-DATE
> Task :wear:mapDebugSourceSetPaths UP-TO-DATE
> Task :wear:generateDebugResources UP-TO-DATE
> Task :wear:mergeDebugResources UP-TO-DATE
> Task :wear:packageDebugResources UP-TO-DATE
> Task :wear:parseDebugLocalResources UP-TO-DATE
> Task :wear:createDebugCompatibleScreenManifests UP-TO-DATE
> Task :wear:extractDeepLinksDebug UP-TO-DATE
> Task :wear:processDebugMainManifest UP

## Summary Table

In [83]:
def status(returncode: int) -> str:
    return "PASS" if returncode == 0 else "FAIL"

summary = f"""
| Module | Status |
|--------|--------|
| ML Pipeline | {status(ml_result.returncode)} |
| Server API | {status(server_result.returncode)} |
| Mobile Unit | {status(mobile_result.returncode)} |
| Wear Unit | {status(wear_result.returncode)} |
"""
display(Markdown(summary))

all_pass = all(r.returncode == 0 for r in [ml_result, server_result, mobile_result, wear_result])
print(f"\n{'='*50}")
print(f"OVERALL: {'ALL TESTS PASSED' if all_pass else 'SOME TESTS FAILED'}")
print(f"{'='*50}")


| Module | Status |
|--------|--------|
| ML Pipeline | PASS |
| Server API | PASS |
| Mobile Unit | PASS |
| Wear Unit | PASS |



OVERALL: ALL TESTS PASSED


In [84]:
# ML coverage
ml_cov = subprocess.run(
    [PYTHON, "-m", "pytest", "tests/", "--cov=src", "--cov-report=term-missing", "--no-header"],
    cwd=str(REPO_ROOT / "ml"),
    capture_output=True, text=True
)
print(ml_cov.stdout[-4000:])


.................                          [ 31%]
tests\test_fl_simulation.py .....                                        [ 38%]
tests\test_loso.py ....                                                  [ 43%]
tests\test_scaler.py ..........                                          [ 56%]
tests\test_tflite_parity.py .......sssss.....                            [ 78%]
tests\test_windowing.py ................                                 [100%]

============================== warnings summary ===============================
tests/test_tflite_parity.py::TestTFLiteModel::test_input_shape
tests/test_tflite_parity.py::TestTFLiteModel::test_output_shape
tests/test_tflite_parity.py::TestTFLiteModel::test_output_deterministic
tests/test_tflite_parity.py::TestKerasVsTFLiteParity::test_output_close
tests/test_tflite_parity.py::TestTrainableModel::test_has_infer_signature
tests/test_tflite_parity.py::TestTrainableModel::test_has_train_signature
tests/test_tflite_parity.py::TestTrainableModel::t

# Evaluation of data

The unit tests above check if the code runs, the next part checks information about how well the entire system actually works, extracting measurable data.

The data obtained is extracted from real artifacts, such as models obtained after training, runs on the WESAD dataset, backend testing on running containers.

| Evidence category | Outputs |
|-----------------------|---------|
| Test environment snapshot | console |
| LOSO generalisation (per-subject) | `loso_per_subject.csv`, `loso_subject_accuracy.png` |
| Hold-out evaluation + FPR/FNR | `holdout_metrics.csv` |
| Operating-threshold + confusion matrix + AUC | `threshold_analysis.csv`, `confusion_matrix.png`, `probability_separation.png`, `threshold_curve.png` |
| Sensor-modality ablation | `sensor_ablation.csv`, `sensor_ablation.png` |
| Scaler parity (Python pkl and Android JSON) | `scaler_parity.csv` |
| Keras, TFLit, INT8 parity | `model_parity.csv`, `keras_vs_tflite_delta.png` |
| On-device latency (feature extract / infer / XAI / end-to-end) | `latency_benchmark.csv`, `latency_distribution.png` |
| Model & memory sizes | `model_sizes.csv` |
| Wearable integration unit-test results | `wearable_unit_tests.csv` |
| Federated-learning convergence + centralised-vs-FL gap + comm cost | `fl_rounds.csv`, `fl_convergence.png`, `fl_comm_cost.csv` |
| Privacy + security validation (k-anon rejection, auth, payload audit) | `privacy_payload_audit.csv`, `security_tests.csv` |
| DB write latency (via live API) | `db_write_latency.csv` |


In [85]:
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ML_DIR = REPO_ROOT / "ml"
if str(ML_DIR) not in sys.path:
    sys.path.insert(0, str(ML_DIR))

MODELS_DIR = ML_DIR / "models"
DATA_NPZ = ML_DIR / "data" / "processed" / "wesad_wrist.npz"

EVAL_DIR = ML_DIR / "evaluation"
RESULTS_DIR = EVAL_DIR / "results"
PLOTS_DIR = EVAL_DIR / "plots"
for d in (RESULTS_DIR, PLOTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 120, "savefig.bbox": "tight", "font.size": 10})

_generated = []

def save_table(df: pd.DataFrame, name: str, desc: str, index: bool = False):
    path = RESULTS_DIR / name
    df.to_csv(path, index=index)
    _generated.append((path, desc))
    print(f"  ↳ wrote {path.relative_to(REPO_ROOT)}  ({len(df)} rows)")
    return path

def save_fig(fig, name: str, desc: str):
    path = PLOTS_DIR / name
    fig.savefig(path)
    plt.close(fig)
    _generated.append((path, desc))
    print(f"  ↳ wrote {path.relative_to(REPO_ROOT)}")
    return path

# load wesad dataset
_d = np.load(DATA_NPZ, allow_pickle=True)
X_raw = _d["X"].astype(np.float32)              
y_true = _d["y"].astype(int) # 0, 1
subject_ids = _d["subject_ids"].astype(str)
FEATURE_NAMES = [str(n) for n in _d["feature_names"]]
N, T, F = X_raw.shape

# apply scaler
_scaler = json.loads((MODELS_DIR / "scaler_params.json").read_text())
SCALER_MEAN = np.asarray(_scaler["mean"], dtype=np.float32)
SCALER_SCALE = np.asarray(_scaler["scale"], dtype=np.float32)
X_scaled = ((X_raw.reshape(-1, F) - SCALER_MEAN) / SCALER_SCALE).reshape(N, T, F).astype(np.float32)

LABEL_MAP = {0: "normal", 1: "stress"}
STRESS_THRESHOLD = 0.85

print(f"Dataset      : {N} windows  |  shape {T}x{F}  |  {len(set(subject_ids))} subjects")
print(f"Class balance: normal={int((y_true==0).sum())}  stress={int((y_true==1).sum())}")
print(f"Features     : {F}  ({', '.join(FEATURE_NAMES[:4])}, ...)")
print(f"Output dirs  : {EVAL_DIR.relative_to(REPO_ROOT)}/{{results,plots}}")


Dataset      : 1738 windows  |  shape 12x23  |  15 subjects
Class balance: normal=1124  stress=614
Features     : 23  (hrv_meanNN, hrv_SDNN, hrv_RMSSD, hrv_pNN50, ...)
Output dirs  : ml\evaluation/{results,plots}


In [86]:
import platform, sys, importlib

def _pkg(name):
    try:
        return importlib.import_module(name).__version__
    except Exception:
        return "n/a"

import numpy as np
_npz = np.load(DATA_NPZ, allow_pickle=True)
_n = len(_npz["X"])
_subjects = sorted(set(_npz["subject_ids"].astype(str)))
_cls = dict(zip(*np.unique(_npz["y"], return_counts=True)))

env_lines = [
    ("Platform", platform.platform()),
    ("Python", sys.version.split()[0]),
    ("TensorFlow", _pkg("tensorflow")),
    ("flwr (Flower)", _pkg("flwr")),
    ("scikit-learn", _pkg("sklearn")),
    ("numpy", _pkg("numpy")),
    ("Dataset", f"WESAD wrist - {_n} windows, {len(_subjects)} subjects ({', '.join(_subjects)})"),
    ("Class balance", f"normal={int(_cls.get(0,0))}  stress={int(_cls.get(1,0))}  ({int(_cls.get(1,0))/_n:.1%} stress)"),
    ("Window shape", f"{_npz['X'].shape[1]} sub-windows × {_npz['X'].shape[2]} features"),
    ("Keras model", str(MODELS_DIR / "mindwave_stress.keras")),
    ("TFLite model (float32)", str(MODELS_DIR / "mindwave_stress.tflite")),
    ("TFLite model (INT8)", str(MODELS_DIR / "mindwave_stress_int8.tflite")),
    ("Server stack", "FastAPI + PostgreSQL + Flower via Docker Compose"),
    ("Evaluation env", "Desktop CPU proxy (AMD/Intel x86-64) - NOT on-device ARM SoC"),
    ("Alert threshold", str(STRESS_THRESHOLD)),
]
for k, v in env_lines:
    print(f"  {k:<26}: {v}")


  Platform                  : Windows-11-10.0.26100-SP0
  Python                    : 3.12.10
  TensorFlow                : 2.21.0
  flwr (Flower)             : 1.30.0
  scikit-learn              : 1.8.0
  numpy                     : 2.4.6
  Dataset                   : WESAD wrist - 1738 windows, 15 subjects (S10, S11, S13, S14, S15, S16, S17, S2, S3, S4, S5, S6, S7, S8, S9)
  Class balance             : normal=1124  stress=614  (35.3% stress)
  Window shape              : 12 sub-windows × 23 features
  Keras model               : c:\Users\daniion6\AndroidStudioProjects\MindWave\ml\models\mindwave_stress.keras
  TFLite model (float32)    : c:\Users\daniion6\AndroidStudioProjects\MindWave\ml\models\mindwave_stress.tflite
  TFLite model (INT8)       : c:\Users\daniion6\AndroidStudioProjects\MindWave\ml\models\mindwave_stress_int8.tflite
  Server stack              : FastAPI + PostgreSQL + Flower via Docker Compose
  Evaluation env            : Desktop CPU proxy (AMD/Intel x86-64) - NOT o

## LOSO Generalisation - Per-Subject Performance

LOSO cross-validation is a way of testing if the model generalises to a new person or not. In this test we extract per-fold results and turn them into a table + a bar chart to analyse the accuracy


In [87]:
loso = json.loads((MODELS_DIR / "loso_metrics.json").read_text())

rows = []
for subj, m in loso["per_fold"].items():
    stress = m["per_class"]["stress"]
    nonstr = m["per_class"]["normal"]
    rows.append({
        "subject": subj,
        "accuracy": round(m["accuracy"], 4),
        "macro_f1": round(m["macro_f1"], 4),
        "weighted_f1": round(m["weighted_f1"], 4),
        "stress_precision": round(stress["precision"], 4),
        "stress_recall": round(stress["recall"], 4),
        "stress_f1": round(stress["f1"], 4),
        "nonstress_f1": round(nonstr["f1"], 4),
        "n_windows": stress["support"] + nonstr["support"],
    })

loso_df = pd.DataFrame(rows).sort_values("subject").reset_index(drop=True)

mean_acc, std_acc = loso["mean_accuracy"], loso["std_accuracy"]
mean_f1, std_f1 = loso["mean_macro_f1"], loso["std_macro_f1"]
summary_row = {
    "subject": "MEAN±STD",
    "accuracy": f"{mean_acc:.4f}±{std_acc:.4f}",
    "macro_f1": f"{mean_f1:.4f}±{std_f1:.4f}",
    "weighted_f1": "", "stress_precision": "", "stress_recall": "",
    "stress_f1": "", "nonstress_f1": "", "n_windows": loso_df["n_windows"].sum(),
}
loso_out = pd.concat([loso_df, pd.DataFrame([summary_row])], ignore_index=True)

print(f"LOSO over {loso['n_folds']} subjects:")
print(f"  mean accuracy = {mean_acc:.1%} ± {std_acc:.1%}")
print(f"  mean macro-F1 = {mean_f1:.1%} ± {std_f1:.1%}")
print(f"  worst subject = {loso_df.loc[loso_df['accuracy'].idxmin(), 'subject']} "
      f"({loso_df['accuracy'].min():.1%}); "
      f"best = {loso_df.loc[loso_df['accuracy'].idxmax(), 'subject']} "
      f"({loso_df['accuracy'].max():.1%})")
save_table(loso_out, "loso_per_subject.csv", "Per-subject LOSO metrics + mean/std summary")

# Per-subject accuracy & macro-F1 with mean band
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(loso_df))
w = 0.4
ax.bar(x - w/2, loso_df["accuracy"], w, label="Accuracy", color="#1f9e9e")
ax.bar(x + w/2, loso_df["macro_f1"], w, label="Macro-F1", color="#7fd4d4")
ax.axhline(mean_acc, color="#0a4f4f", ls="--", lw=1.2, label=f"Mean acc {mean_acc:.2f}")
ax.axhspan(mean_acc - std_acc, mean_acc + std_acc, color="#0a4f4f", alpha=0.08)
ax.set_xticks(x); ax.set_xticklabels(loso_df["subject"], rotation=45)
ax.set_ylim(0.5, 1.02); ax.set_ylabel("Score")
ax.set_title("LOSO per-subject performance (left-out subject = unseen user)")
ax.legend(loc="lower right", ncol=2, fontsize=8)
save_fig(fig, "loso_subject_accuracy.png", "Per-subject LOSO accuracy & macro-F1 bar chart")
loso_out


LOSO over 15 subjects:
  mean accuracy = 95.2% ± 4.1%
  mean macro-F1 = 94.7% ± 4.5%
  worst subject = S15 (87.3%); best = S11 (100.0%)
  ↳ wrote ml\evaluation\results\loso_per_subject.csv  (16 rows)
  ↳ wrote ml\evaluation\plots\loso_subject_accuracy.png


,subject,accuracy,macro_f1,weighted_f1,stress_precision,stress_recall,stress_f1,nonstress_f1,n_windows
0,S10,0.8926,0.8895,0.8942,0.7857,0.9778,0.8713,0.9078,121
1,S11,1.0,1.0,1.0,1.0,1.0,1.0,1.0,117
2,S13,1.0,1.0,1.0,1.0,1.0,1.0,1.0,117
3,S14,0.906,0.9011,0.9073,0.8163,0.9524,0.8791,0.9231,117
4,S15,0.8729,0.859,0.8718,0.8462,0.7857,0.8148,0.9032,118
5,S16,1.0,1.0,1.0,1.0,1.0,1.0,1.0,116
6,S17,0.9174,0.9066,0.9147,1.0,0.7778,0.875,0.9383,121
7,S2,0.9364,0.9249,0.9345,1.0,0.8108,0.8955,0.9542,110
8,S3,0.9459,0.9414,0.9462,0.9024,0.9487,0.925,0.9577,111
9,S4,1.0,1.0,1.0,1.0,1.0,1.0,1.0,113


In [88]:
holdout = json.loads((MODELS_DIR / "holdout_metrics.json").read_text())

# Build a structured table mirroring the LOSO one
stress_h = holdout["per_class"]["stress"]
nonstr_h = holdout["per_class"]["normal"]
n_ho = stress_h["support"] + nonstr_h["support"]

ho_stress_recall = stress_h["recall"]
ho_nonstress_recall = nonstr_h["recall"]

ho_rows = [
    {"split": "hold-out (all-subject model)",
     "n_windows": n_ho,
     "accuracy": round(holdout["accuracy"], 4),
     "macro_f1": round(holdout["macro_f1"], 4),
     "stress_precision": round(stress_h["precision"], 4),
     "stress_recall": round(stress_h["recall"], 4),
     "stress_f1": round(stress_h["f1"], 4),
     "fpr": round(1.0 - nonstr_h["recall"], 4),
     "fnr": round(1.0 - stress_h["recall"], 4),
    },
    {"split": "LOSO (cross-subject, mean)",
     "n_windows": loso_df["n_windows"].sum(),
     "accuracy": round(mean_acc, 4),
     "macro_f1": round(mean_f1, 4),
     "stress_precision": round(loso_df["stress_precision"].mean(), 4),
     "stress_recall": round(loso_df["stress_recall"].mean(), 4),
     "stress_f1": round(loso_df["stress_f1"].mean(), 4),
     "fpr": round(1.0 - loso_df["nonstress_f1"].mean(), 4),
     "fnr": round(1.0 - loso_df["stress_recall"].mean(), 4),
    },
]
ho_df = pd.DataFrame(ho_rows)
print("ML evaluation summary (hold-out vs LOSO):")
print(ho_df[["split","accuracy","macro_f1","stress_recall","fpr","fnr"]].to_string(index=False))
save_table(ho_df, "holdout_metrics.csv", "Hold-out and LOSO evaluation including FPR/FNR")
ho_df


ML evaluation summary (hold-out vs LOSO):
                       split  accuracy  macro_f1  stress_recall    fpr    fnr
hold-out (all-subject model)    0.9957    0.9953         0.9880 0.0000 0.0120
  LOSO (cross-subject, mean)    0.9520    0.9474         0.9335 0.0374 0.0665
  ↳ wrote ml\evaluation\results\holdout_metrics.csv  (2 rows)


,split,n_windows,accuracy,macro_f1,stress_precision,stress_recall,stress_f1,fpr,fnr
0,hold-out (all-subject model),233,0.9957,0.9953,1.0000,0.9880,0.9939,0.0000,0.0120
1,"LOSO (cross-subject, mean)",1738,0.9520,0.9474,0.9383,0.9335,0.9322,0.0374,0.0665


## Operating-Threshold Analysis

The deployed system raises an alert when the probability of stress > 0.85. That number is a design decision, which could impact the number of notifications a user may get - too low and there could be false alarms, too high and there would be missed stress episodes.

In this scenario, we go across different threshold {0.50, 0.70, 0.85, 0.90} and report relevant data like precision, recall etc. to identify the differences between the thresholds.


In [89]:
import tensorflow as tf
from sklearn.metrics import (
    precision_recall_fscore_support, accuracy_score, confusion_matrix,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
)

print("Loading Keras model (one-time)…")
keras_model = tf.keras.models.load_model(str(MODELS_DIR / "mindwave_stress.keras"))

proba = keras_model.predict(X_scaled, batch_size=256, verbose=0)
p_stress = proba[:, 1] if proba.ndim == 2 and proba.shape[1] == 2 else proba.ravel()

# Threshold-independent discriminative power
roc_auc = roc_auc_score(y_true, p_stress)
pr_auc = average_precision_score(y_true, p_stress)
print(f"ROC-AUC = {roc_auc:.4f}   PR-AUC = {pr_auc:.4f}   (in-sample; LOSO acc = 93.7% in 2.1)")

THRESHOLDS = [0.50, 0.70, 0.85, 0.90]
trows = []
for thr in THRESHOLDS:
    y_hat = (p_stress > thr).astype(int)
    pr, rc, f1, _ = precision_recall_fscore_support(
        y_true, y_hat, average="binary", pos_label=1, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0, 1]).ravel()
    trows.append({
        "threshold": thr,
        "accuracy": round(accuracy_score(y_true, y_hat), 4),
        "stress_precision": round(pr, 4),
        "stress_recall": round(rc, 4),
        "stress_f1": round(f1, 4),
        "true_pos": int(tp), "false_pos": int(fp),
        "false_neg": int(fn), "true_neg": int(tn),
    })
thr_df = pd.DataFrame(trows)
save_table(thr_df, "threshold_analysis.csv",
           f"In-sample precision/recall/F1 vs alert threshold (ROC-AUC={roc_auc:.4f}, PR-AUC={pr_auc:.4f})")

# Probability margin histogram
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(p_stress[y_true == 0], bins=40, range=(0, 1), color="#1f9e9e",
        alpha=0.75, label="True normal")
ax.hist(p_stress[y_true == 1], bins=40, range=(0, 1), color="#e07a3f",
        alpha=0.75, label="True stress")
ax.axvline(STRESS_THRESHOLD, color="black", ls="--", lw=1.4, label=f"Alert threshold {STRESS_THRESHOLD}")
ax.set_xlabel("P(stress)"); ax.set_ylabel("Window count (log)"); ax.set_yscale("log")
ax.set_title("Predicted-probability separation by true class")
ax.legend(fontsize=8)
save_fig(fig, "probability_separation.png", "Histogram of P(stress) split by true label")

# Sweep threshold to plot precision/recall/F1 curves and visualize the trade-off
sweep = np.linspace(0.05, 0.95, 91)
P, R, Fb = [], [], []
for t in sweep:
    yh = (p_stress > t).astype(int)
    p_, r_, f_, _ = precision_recall_fscore_support(
        y_true, yh, average="binary", pos_label=1, zero_division=0)
    P.append(p_); R.append(r_); Fb.append(f_)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sweep, P, label="Precision", color="#1f9e9e")
ax.plot(sweep, R, label="Recall", color="#e07a3f")
ax.plot(sweep, Fb, label="F1", color="#0a4f4f")
ax.axvline(STRESS_THRESHOLD, color="grey", ls="--", lw=1, label=f"Prod threshold {STRESS_THRESHOLD}")
ax.set_xlabel("Decision threshold on P(stress)"); ax.set_ylabel("Score")
ax.set_title("Operating-point trade-off (stress class, in-sample)")
ax.legend(loc="lower center", ncol=2, fontsize=8); ax.set_ylim(0, 1.02)
save_fig(fig, "threshold_curve.png", "Precision/recall/F1 vs decision threshold")

# Confusion matrix
y_hat85 = (p_stress > STRESS_THRESHOLD).astype(int)
cm = confusion_matrix(y_true, y_hat85, labels=[0, 1])
cm_norm = cm / cm.sum(axis=1, keepdims=True).clip(min=1)
fig, ax = plt.subplots(figsize=(4.2, 3.6))
im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i,j]}\n({cm_norm[i,j]:.0%})", ha="center", va="center",
                color="white" if cm_norm[i, j] > 0.5 else "black", fontsize=10)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["normal", "stress"]); ax.set_yticklabels(["normal", "stress"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title(f"Confusion matrix @ threshold {STRESS_THRESHOLD}")
save_fig(fig, "confusion_matrix.png", f"Confusion matrix at production threshold {STRESS_THRESHOLD}")
thr_df

Loading Keras model (one-time)…
ROC-AUC = 1.0000   PR-AUC = 1.0000   (in-sample; LOSO acc = 93.7% in 2.1)
  ↳ wrote ml\evaluation\results\threshold_analysis.csv  (4 rows)
  ↳ wrote ml\evaluation\plots\probability_separation.png
  ↳ wrote ml\evaluation\plots\threshold_curve.png
  ↳ wrote ml\evaluation\plots\confusion_matrix.png


,threshold,accuracy,stress_precision,stress_recall,stress_f1,true_pos,false_pos,false_neg,true_neg
0,0.50,0.9994,0.9984,1.0,0.9992,614,1,0,1123
1,0.70,0.9994,0.9984,1.0,0.9992,614,1,0,1123
2,0.85,0.9994,0.9984,1.0,0.9992,614,1,0,1123
3,0.90,0.9994,0.9984,1.0,0.9992,614,1,0,1123


## Sensor-Modality Ablation

Through this test, we want to check which wrist sensors actually *drive* the decision. We mask each group by replacing 0 to major feature-related data (HRV from BVP, EDA, skin temp, accelerometer), then measure how much accuracy and recall the shipped model loses. A large model geniunely indicated that we need a multi-modal approach, like we currently have.

In [90]:
GROUPS = {
    "HRV (BVP)": [i for i, n in enumerate(FEATURE_NAMES) if n.startswith("hrv_")],
    "EDA":       [i for i, n in enumerate(FEATURE_NAMES) if n.startswith("eda_")],
    "TEMP":      [i for i, n in enumerate(FEATURE_NAMES) if n.startswith("temp_")],
    "ACC":       [i for i, n in enumerate(FEATURE_NAMES) if n.startswith("acc_")],
}

def _eval(Xin):
    pr = keras_model.predict(Xin, batch_size=256, verbose=0)
    ps = pr[:, 1] if pr.ndim == 2 and pr.shape[1] == 2 else pr.ravel()
    yh = (ps > STRESS_THRESHOLD).astype(int)
    acc = accuracy_score(y_true, yh)
    _, rec, _, _ = precision_recall_fscore_support(
        y_true, yh, average="binary", pos_label=1, zero_division=0)
    return acc, rec

base_acc, base_rec = _eval(X_scaled)
arows = [{"masked_modality": "none (baseline)", "n_features": 0,
          "accuracy": round(base_acc, 4), "stress_recall": round(base_rec, 4),
          "accuracy_drop": 0.0}]
for name, idx in GROUPS.items():
    Xm = X_scaled.copy()
    Xm[:, :, idx] = 0.0          
    a, r = _eval(Xm)
    arows.append({
        "masked_modality": name, "n_features": len(idx),
        "accuracy": round(a, 4), "stress_recall": round(r, 4),
        "accuracy_drop": round(base_acc - a, 4),
    })
abl_df = pd.DataFrame(arows)
most = abl_df.iloc[1:].sort_values("accuracy_drop", ascending=False).iloc[0]
print(f"Baseline accuracy = {base_acc:.1%}.  Most critical modality: "
      f"{most['masked_modality']} (−{most['accuracy_drop']:.1%} when removed)")
save_table(abl_df, "sensor_ablation.csv", "Accuracy / stress-recall when each sensor group is masked")

# Plot: accuracy drop per masked modality
abl_plot = abl_df.iloc[1:]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(abl_plot["masked_modality"], abl_plot["accuracy_drop"], color="#e07a3f")
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("Accuracy drop vs baseline"); ax.set_xlabel("Masked sensor modality")
ax.set_title(f"Sensor ablation - baseline accuracy {base_acc:.1%}")
for b, v in zip(bars, abl_plot["accuracy_drop"]):
    ax.text(b.get_x() + b.get_width()/2, v, f"{v:.1%}", ha="center",
            va="bottom" if v >= 0 else "top", fontsize=9)
save_fig(fig, "sensor_ablation.png", "Accuracy drop when each sensor modality is masked")
abl_df


Baseline accuracy = 99.9%.  Most critical modality: EDA (−6.7% when removed)
  ↳ wrote ml\evaluation\results\sensor_ablation.csv  (5 rows)
  ↳ wrote ml\evaluation\plots\sensor_ablation.png


,masked_modality,n_features,accuracy,stress_recall,accuracy_drop
0,none (baseline),0,0.9994,1.0000,0.0000
1,HRV (BVP),7,0.9609,0.9121,0.0386
2,EDA,7,0.9321,0.8795,0.0673
3,TEMP,5,0.9528,0.9723,0.0466
4,ACC,4,0.9764,0.9397,0.0230


## Deployment parity between Keras, TFLite and INT8

The model used by the phone is a converted .tflite file, not the Keras model trained on PC. If conversion changed the maths, on-device alerts would diverge from the validated model. In this evaluation step, we feed the same data through 3 different models to determine the classification agreement at 0.85 threshold.

In [91]:
PARITY_N = min(400, N)
rng = np.random.default_rng(42)
sel = rng.choice(N, size=PARITY_N, replace=False)
X_par = X_scaled[sel]
p_keras = p_stress[sel]                     

def tflite_stress_probs(model_path, X):
    """Return P(stress) per window from a TFLite model (handles int8 I/O)."""
    interp = tf.lite.Interpreter(model_path=str(model_path))
    interp.allocate_tensors()
    inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
    probs = np.empty(len(X), dtype=np.float32)
    for i, x in enumerate(X):
        xi = x[np.newaxis].astype(np.float32)
        if inp["dtype"] == np.int8:         
            s, z = inp["quantization"]
            xi = np.clip(np.round(xi / s + z), -128, 127).astype(np.int8)
        interp.set_tensor(inp["index"], xi)
        interp.invoke()
        o = interp.get_tensor(out["index"])[0].astype(np.float32)
        if out["dtype"] == np.int8:           
            s, z = out["quantization"]
            o = (o - z) * s
        probs[i] = o[1] if o.shape[-1] == 2 else o.ravel()[0]
    return probs

parity_rows = []
delta_store = {}
for tag, fname in [("TFLite float32", "mindwave_stress.tflite"),
                   ("TFLite INT8", "mindwave_stress_int8.tflite")]:
    fpath = MODELS_DIR / fname
    if not fpath.exists():
        print(f"  (skipped {tag}: {fname} not found)"); continue
    try:
        p_lite = tflite_stress_probs(fpath, X_par)
    except Exception as exc:
        print(f"  (skipped {tag}: {type(exc).__name__}: {exc})"); continue
    delta = np.abs(p_keras - p_lite)
    delta_store[tag] = delta
    agree = (np.array((p_keras > STRESS_THRESHOLD), int)
             == np.array((p_lite > STRESS_THRESHOLD), int)).mean()
    parity_rows.append({
        "model": tag, "n_windows": PARITY_N,
        "mean_abs_delta": float(f"{delta.mean():.3e}"),
        "max_abs_delta": float(f"{delta.max():.3e}"),
        "p95_abs_delta": float(f"{np.percentile(delta, 95):.3e}"),
        "classification_agreement": round(float(agree), 4),
    })

parity_df = pd.DataFrame(parity_rows)
for _, r in parity_df.iterrows():
    print(f"{r['model']:16s}  mean|Δ|={r['mean_abs_delta']:.2e}  "
          f"max|Δ|={r['max_abs_delta']:.2e}  agreement@0.85={r['classification_agreement']:.1%}")
save_table(parity_df, "model_parity.csv",
           "Keras vs TFLite/INT8 probability delta and classification agreement @0.85")

# --- Plot: probability-delta distribution per converted model ---
if delta_store:
    fig, ax = plt.subplots(figsize=(8, 4))
    colors = {"TFLite float32": "#1f9e9e", "TFLite INT8": "#e07a3f"}
    for tag, delta in delta_store.items():
        nz = np.clip(delta, 1e-9, None)
        ax.hist(nz, bins=np.logspace(-9, 0, 40), alpha=0.65,
                label=f"{tag} (mean {delta.mean():.1e})", color=colors.get(tag))
    ax.set_xscale("log"); ax.set_xlabel("|P_keras − P_tflite|  (log)")
    ax.set_ylabel("Window count")
    ax.set_title("Deployment parity: per-window probability delta vs Keras reference")
    ax.legend(fontsize=8)
    save_fig(fig, "keras_vs_tflite_delta.png", "Probability-delta histogram (Keras vs TFLite/INT8)")
parity_df


TFLite float32    mean|Δ|=3.85e-10  max|Δ|=7.82e-08  agreement@0.85=100.0%
TFLite INT8       mean|Δ|=1.97e-05  max|Δ|=1.68e-03  agreement@0.85=100.0%
  ↳ wrote ml\evaluation\results\model_parity.csv  (2 rows)
  ↳ wrote ml\evaluation\plots\keras_vs_tflite_delta.png


,model,n_windows,mean_abs_delta,max_abs_delta,p95_abs_delta,classification_agreement
0,TFLite float32,400,3.852000e-10,7.823000e-08,7.276000e-11,1.0
1,TFLite INT8,400,1.971000e-05,1.675000e-03,9.135000e-06,1.0


In [92]:
# Scaler parity: compare Python scaler (pkl) vs Android JSON parameters
import joblib

sc_pkl = joblib.load(MODELS_DIR / "scaler.pkl")
sc_json = json.loads((MODELS_DIR / "scaler_params.json").read_text())

pkl_mean = np.asarray(sc_pkl.mean_, dtype=np.float64)
pkl_scale = np.asarray(sc_pkl.scale_, dtype=np.float64)
json_mean = np.asarray(sc_json["mean"], dtype=np.float64)
json_scale = np.asarray(sc_json["scale"], dtype=np.float64)

delta_mean = np.abs(pkl_mean - json_mean)
delta_scale = np.abs(pkl_scale - json_scale)
nrm_mean = delta_mean / (np.abs(pkl_mean).clip(1e-9))
nrm_scale = delta_scale / (np.abs(pkl_scale).clip(1e-9))

sp_rows = []
for i, feat in enumerate(FEATURE_NAMES):
    sp_rows.append({
        "feature": feat,
        "pkl_mean": round(float(pkl_mean[i]), 6), "json_mean": round(float(json_mean[i]), 6),
        "delta_mean": float(f"{delta_mean[i]:.2e}"),
        "pkl_scale": round(float(pkl_scale[i]), 6), "json_scale": round(float(json_scale[i]), 6),
        "delta_scale": float(f"{delta_scale[i]:.2e}"),
        "normalised_mean_err": float(f"{nrm_mean[i]:.2e}"),
    })
sp_df = pd.DataFrame(sp_rows)
verdict = "PASS" if (delta_mean.max() == 0 and delta_scale.max() == 0) else "MISMATCH"
print(f"Scaler parity: {verdict}")
print(f"  max |Δ mean|  = {delta_mean.max():.2e}   mean |Δ mean|  = {delta_mean.mean():.2e}")
print(f"  max |Δ scale| = {delta_scale.max():.2e}   mean |Δ scale| = {delta_scale.mean():.2e}")
print(f"  allclose (atol=1e-6): mean={np.allclose(pkl_mean, json_mean, atol=1e-6)}, "
      f"scale={np.allclose(pkl_scale, json_scale, atol=1e-6)}")
save_table(sp_df, "scaler_parity.csv",
           f"Per-feature scaler parity: Python pkl vs Android JSON - {verdict}")
sp_df.head(6)


Scaler parity: PASS
  max |Δ mean|  = 0.00e+00   mean |Δ mean|  = 0.00e+00
  max |Δ scale| = 0.00e+00   mean |Δ scale| = 0.00e+00
  allclose (atol=1e-6): mean=True, scale=True
  ↳ wrote ml\evaluation\results\scaler_parity.csv  (23 rows)


,feature,pkl_mean,json_mean,delta_mean,pkl_scale,json_scale,delta_scale,normalised_mean_err
0,hrv_meanNN,830.567588,830.567588,0.0,176.583916,176.583916,0.0,0.0
1,hrv_SDNN,197.972214,197.972214,0.0,148.156243,148.156243,0.0,0.0
2,hrv_RMSSD,272.130577,272.130577,0.0,214.407481,214.407481,0.0,0.0
3,hrv_pNN50,70.456176,70.456176,0.0,33.606496,33.606496,0.0,0.0
4,hrv_LF,0.032477,0.032477,0.0,0.015531,0.015531,0.0,0.0
5,hrv_HF,0.093362,0.093362,0.0,0.039485,0.039485,0.0,0.0


## Inference & XAI Latency

At this step, we benchmark, on the PC's CPU, the latency associated to TFLite inference + vanilla-gradient XAI explanation.


In [93]:
from src.xai import gradient_feature_importance

REPS = 200
warmup = 10
sel_lat = rng.choice(N, size=REPS + warmup, replace=False)
Xlat = X_scaled[sel_lat]

def bench(fn, X):
    times = []
    for i, x in enumerate(X):
        t0 = time.perf_counter()
        fn(x)
        dt = (time.perf_counter() - t0) * 1000.0
        if i >= warmup:
            times.append(dt)
    return np.asarray(times)

_interp = tf.lite.Interpreter(model_path=str(MODELS_DIR / "mindwave_stress.tflite"))
_interp.allocate_tensors()
_in, _out = _interp.get_input_details()[0], _interp.get_output_details()[0]
def _tflite_infer(x):
    _interp.set_tensor(_in["index"], x[np.newaxis].astype(np.float32))
    _interp.invoke()
    return _interp.get_tensor(_out["index"])

def _xai(x):
    return gradient_feature_importance(keras_model, x.astype(np.float32))

bench_specs = [
    ("TFLite infer (batch=1)", _tflite_infer),
    ("XAI vanilla-gradient", _xai),
]
lrows = []
lat_store = {}
for name, fn in bench_specs:
    t = bench(fn, Xlat)
    lat_store[name] = t
    lrows.append({
        "operation": name, "reps": len(t),
        "mean_ms": round(t.mean(), 3), "p50_ms": round(np.percentile(t, 50), 3),
        "p95_ms": round(np.percentile(t, 95), 3), "p99_ms": round(np.percentile(t, 99), 3),
        "max_ms": round(t.max(), 3),
    })
lat_df = pd.DataFrame(lrows)
total_p95 = lat_df["p95_ms"].sum()
print(f"Per-window p95: infer + XAI = {total_p95:.2f} ms  "
      f"(R7 budget = 10000 ms → {total_p95/10000:.3%} used)")
save_table(lat_df, "latency_benchmark.csv",
           "Desktop CPU latency (ms) for TFLite inference and vanilla-gradient XAI, p50/p95/p99")

# Latency Distribution
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot([lat_store[k] for k in lat_store], labels=list(lat_store.keys()),
           showfliers=False, patch_artist=True,
           boxprops=dict(facecolor="#7fd4d4"), medianprops=dict(color="#0a4f4f"))
ax.set_ylabel("Latency per window (ms)")
ax.set_title("On-device compute latency (desktop CPU proxy, batch=1)")
save_fig(fig, "latency_distribution.png", "Boxplot of per-window inference and XAI latency")
lat_df


Per-window p95: infer + XAI = 246.80 ms  (R7 budget = 10000 ms → 2.468% used)
  ↳ wrote ml\evaluation\results\latency_benchmark.csv  (2 rows)
  ↳ wrote ml\evaluation\plots\latency_distribution.png


C:\Users\daniion6\AppData\Local\Temp\ipykernel_43264\2398694264.py:53: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  ax.boxplot([lat_store[k] for k in lat_store], labels=list(lat_store.keys()),


,operation,reps,mean_ms,p50_ms,p95_ms,p99_ms,max_ms
0,TFLite infer (batch=1),200,0.069,0.068,0.078,0.104,0.162
1,XAI vanilla-gradient,200,163.494,144.267,246.720,319.996,1853.091


In [94]:
# Feature-extraction latency + end-to-end alert latency
from src.features import subwindow_feature_vector, hrv_freq_features, scr_peak_features
from src.preprocessing import filter_bvp, filter_eda, filter_temp, filter_acc_mag, acc_magnitude
from src.config import WRIST_FS, SUBWINDOW_SECONDS, N_SUBWINDOWS

# Synthetic 60s raw-sensor buffers at WESAD E4 wrist sampling rates
_rng2 = np.random.default_rng(99)
BVP_N  = WRIST_FS["BVP"] * 60        # 3840 samples @ 64 Hz
EDA_N  = WRIST_FS["EDA"] * 60        # 240 samples @ 4 Hz
TEMP_N = WRIST_FS["TEMP"] * 60       # 240 samples @ 4 Hz
ACC_N  = WRIST_FS["ACC"] * 60        # 1920 samples @ 32 Hz

def _make_raw():
    bvp  = (_rng2.standard_normal(BVP_N) * 200 + 600).astype(np.float32)
    eda  = (_rng2.standard_normal(EDA_N) * 0.5 + 2).astype(np.float32)
    temp = (_rng2.standard_normal(TEMP_N) * 0.3 + 33).astype(np.float32)
    acc  = (_rng2.standard_normal((ACC_N, 3)) * 0.1).astype(np.float32)
    return bvp, eda, temp, acc

def _feature_extraction_pipeline(raw):
    bvp, eda, temp, acc = raw
    bvp_f  = filter_bvp(bvp, WRIST_FS["BVP"])
    eda_f  = filter_eda(eda, WRIST_FS["EDA"])
    temp_f = filter_temp(temp)
    acc_m  = filter_acc_mag(acc_magnitude(acc), WRIST_FS["ACC"])
    hrv_freq = hrv_freq_features(bvp_f, WRIST_FS["BVP"])
    scr_pk   = scr_peak_features(eda_f, WRIST_FS["EDA"])
    win = np.zeros((N_SUBWINDOWS, F), dtype=np.float32)
    for k in range(N_SUBWINDOWS):
        s = k * SUBWINDOW_SECONDS
        bvp_sw  = bvp_f[int(s*WRIST_FS["BVP"]):int((s+SUBWINDOW_SECONDS)*WRIST_FS["BVP"])]
        eda_sw  = eda_f[int(s*WRIST_FS["EDA"]):int((s+SUBWINDOW_SECONDS)*WRIST_FS["EDA"])]
        temp_sw = temp_f[int(s*WRIST_FS["TEMP"]):int((s+SUBWINDOW_SECONDS)*WRIST_FS["TEMP"])]
        acc_sw  = acc_m[int(s*WRIST_FS["ACC"]):int((s+SUBWINDOW_SECONDS)*WRIST_FS["ACC"])]
        win[k] = subwindow_feature_vector(bvp_sw, eda_sw, temp_sw, acc_sw, hrv_freq, scr_pk)
    return win

def _e2e(raw):
    """Full pipeline: feature extraction → scale → TFLite infer → threshold check."""
    win = _feature_extraction_pipeline(raw)
    win_scaled = ((win.reshape(-1, F) - SCALER_MEAN) / SCALER_SCALE).reshape(1, N_SUBWINDOWS, F).astype(np.float32)
    _interp.set_tensor(_in["index"], win_scaled)
    _interp.invoke()
    p = _interp.get_tensor(_out["index"])[0]
    return float(p[1] if len(p) == 2 else p[0]) > STRESS_THRESHOLD

REPS_FE  = 50
WARMUP_FE = 5
raw_samples = [_make_raw() for _ in range(REPS_FE + WARMUP_FE)]

def _bench_fn(fn, samples, warmup):
    ts = []
    for i, s in enumerate(samples):
        t0 = time.perf_counter()
        fn(s)
        dt = (time.perf_counter() - t0) * 1000.0
        if i >= warmup:
            ts.append(dt)
    return np.asarray(ts)

print("Benchmarking feature extraction (50 reps)…")
t_fe  = _bench_fn(_feature_extraction_pipeline, raw_samples, WARMUP_FE)
t_e2e = _bench_fn(_e2e, raw_samples, WARMUP_FE)

fe_row  = {"operation": "Feature extraction (60s raw → 12×23 tensor)",
           "reps": len(t_fe),
           "mean_ms": round(t_fe.mean(), 1), "p50_ms": round(np.percentile(t_fe, 50), 1),
           "p95_ms": round(np.percentile(t_fe, 95), 1), "p99_ms": round(np.percentile(t_fe, 99), 1),
           "max_ms": round(t_fe.max(), 1)}
e2e_row = {"operation": "End-to-end alert (feat→scale→infer→threshold)",
           "reps": len(t_e2e),
           "mean_ms": round(t_e2e.mean(), 1), "p50_ms": round(np.percentile(t_e2e, 50), 1),
           "p95_ms": round(np.percentile(t_e2e, 95), 1), "p99_ms": round(np.percentile(t_e2e, 99), 1),
           "max_ms": round(t_e2e.max(), 1)}

full_lat = pd.concat([lat_df, pd.DataFrame([fe_row, e2e_row])], ignore_index=True)
budget_p95 = full_lat.loc[full_lat.operation.str.contains("End-to-end"), "p95_ms"].iloc[0]
print(f"\nEnd-to-end alert p95 = {budget_p95:.1f} ms  vs R7 budget = 10 000 ms  ({budget_p95/10000:.2%} used)")
save_table(full_lat, "latency_benchmark.csv",
           "Desktop CPU latency all pipeline stages: p50/p95/p99 (overwrites previous partial table)")

# Plot all stages together
labels = [r["operation"].split("(")[0].strip() for _, r in full_lat.iterrows()]
p50s   = full_lat["p50_ms"].values
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(range(len(p50s)), p50s, color="#1f9e9e")
ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("p50 latency (ms)"); ax.set_title("Pipeline latency breakdown (desktop CPU proxy, batch=1)")
for b, v in zip(bars, p50s):
    ax.text(v + 0.3, b.get_y() + b.get_height()/2, f"{v:.1f} ms", va="center", fontsize=8)
save_fig(fig, "latency_distribution.png", "Latency p50 per pipeline stage")
full_lat


Benchmarking feature extraction (50 reps)…

End-to-end alert p95 = 106.6 ms  vs R7 budget = 10 000 ms  (1.07% used)
  ↳ wrote ml\evaluation\results\latency_benchmark.csv  (4 rows)
  ↳ wrote ml\evaluation\plots\latency_distribution.png


,operation,reps,mean_ms,p50_ms,p95_ms,p99_ms,max_ms
0,TFLite infer (batch=1),200,0.069,0.068,0.078,0.104,0.162
1,XAI vanilla-gradient,200,163.494,144.267,246.720,319.996,1853.091
2,Feature extraction (60s raw → 12×23 tensor),50,64.300,58.500,99.400,144.000,146.400
3,End-to-end alert (feat→scale→infer→threshold),50,70.800,60.100,106.600,222.100,278.700


In [95]:
# Model comparison
MODEL_FILES = {
    "mindwave_stress.keras": "Training reference (Keras)",
    "mindwave_stress.tflite": "Deployed - float32 TFLite",
    "mindwave_stress_int8.tflite": "Deployed - INT8 quantised TFLite",
    "mindwave_stress_trainable.tflite": "On-device FL training (5-signature TFLite)",
}
size_rows = []
for fname, desc in MODEL_FILES.items():
    p = MODELS_DIR / fname
    if not p.exists():
        continue
    sz = p.stat().st_size
    size_rows.append({"model": fname, "description": desc,
                      "size_bytes": sz, "size_kb": round(sz/1024, 1),
                      "size_mb": round(sz/1024/1024, 3)})

# Keras weight tensor byte count (actual parameter memory)
import tensorflow as tf
km = tf.keras.models.load_model(str(MODELS_DIR / "mindwave_stress.keras"))
w = km.get_weights()
param_bytes = sum(a.nbytes for a in w)
param_count = sum(a.size for a in w)
size_rows.append({"model": "mindwave_stress.keras (weights only)", "description": "In-memory parameter budget",
                  "size_bytes": param_bytes, "size_kb": round(param_bytes/1024, 1),
                  "size_mb": round(param_bytes/1024/1024, 3)})
del km, w

sz_df = pd.DataFrame(size_rows)
print(f"Total trainable parameters : {param_count:,}")
print(f"Model weight memory (fp32) : {param_bytes/1024:.1f} KB")
tflite_float = sz_df.loc[sz_df.model=="mindwave_stress.tflite", "size_kb"].iloc[0]
tflite_int8  = sz_df.loc[sz_df.model=="mindwave_stress_int8.tflite", "size_kb"].iloc[0]
print(f"Quantisation size reduction: {tflite_float:.1f} KB → {tflite_int8:.1f} KB "
      f"({(1-tflite_int8/tflite_float):.1%} smaller)")
save_table(sz_df, "model_sizes.csv", "Model file sizes and in-memory weight budget")
sz_df


Total trainable parameters : 36,066
Model weight memory (fp32) : 140.9 KB
Quantisation size reduction: 232.7 KB → 138.2 KB (40.6% smaller)
  ↳ wrote ml\evaluation\results\model_sizes.csv  (5 rows)


,model,description,size_bytes,size_kb,size_mb
0,mindwave_stress.keras,Training reference (Keras),484431,473.1,0.462
1,mindwave_stress.tflite,Deployed - float32 TFLite,238316,232.7,0.227
2,mindwave_stress_int8.tflite,Deployed - INT8 quantised TFLite,141512,138.2,0.135
3,mindwave_stress_trainable.tflite,On-device FL training (5-signature TFLite),923844,902.2,0.881
4,mindwave_stress.keras (weights only),In-memory parameter budget,144264,140.9,0.138


## Federated-Learning Convergence

In order for privacy to be conserved, the FL arch must be working, therefore the global model assembled should improve round over round. For this test, we read the flower simulation history and chart distributed-eval acc and loss per round. A rising accuracy curve is proof that the strat we currently use (FedAvg) is helping the global model learn.


In [96]:
fl_hist = json.loads((MODELS_DIR / "fl_global_smoke.history.json").read_text())

def _as_map(pairs):
    return {int(r): float(v) for r, v in pairs} if pairs else {}

eval_acc = _as_map(fl_hist.get("metrics_distributed", {}).get("accuracy"))
eval_loss = _as_map(fl_hist.get("losses_distributed"))
fit_acc = _as_map(fl_hist.get("metrics_distributed_fit", {}).get("accuracy"))
fit_loss = _as_map(fl_hist.get("metrics_distributed_fit", {}).get("loss"))

rounds = sorted(set(eval_acc) | set(eval_loss) | set(fit_acc))
fl_rows = [{
    "round": r,
    "fit_accuracy": round(fit_acc.get(r, float("nan")), 4),
    "fit_loss": round(fit_loss.get(r, float("nan")), 4),
    "eval_accuracy": round(eval_acc.get(r, float("nan")), 4),
    "eval_loss": round(eval_loss.get(r, float("nan")), 4),
} for r in rounds]
fl_df = pd.DataFrame(fl_rows)
if eval_acc:
    first_r, last_r = rounds[0], rounds[-1]
    print(f"FL distributed-eval accuracy: round {first_r} = {eval_acc.get(first_r, float('nan')):.1%}"
          f" → round {last_r} = {eval_acc.get(last_r, float('nan')):.1%}"
          f"  (+{(eval_acc.get(last_r,0)-eval_acc.get(first_r,0)):.1%} over {len(rounds)} rounds)")
save_table(fl_df, "fl_rounds.csv", "Federated-learning per-round fit/eval accuracy and loss")

fig, (axa, axl) = plt.subplots(1, 2, figsize=(11, 4))
if fit_acc:
    axa.plot(list(fit_acc), list(fit_acc.values()), "o-", color="#7fd4d4", label="fit (train)")
if eval_acc:
    axa.plot(list(eval_acc), list(eval_acc.values()), "o-", color="#0a4f4f", label="distributed eval")
axa.set_xlabel("FL round"); axa.set_ylabel("Accuracy"); axa.set_ylim(0, 1.02)
axa.set_title("FedAvg accuracy by round"); axa.legend(fontsize=8)
axa.set_xticks(rounds)
if fit_loss:
    axl.plot(list(fit_loss), list(fit_loss.values()), "o-", color="#e07a3f", label="fit loss")
if eval_loss:
    axl.plot(list(eval_loss), list(eval_loss.values()), "o-", color="#b04a1f", label="eval loss")
axl.set_xlabel("FL round"); axl.set_ylabel("Loss")
axl.set_title("FedAvg loss by round"); axl.legend(fontsize=8); axl.set_xticks(rounds)
save_fig(fig, "fl_convergence.png", "Federated-learning accuracy and loss per round")
fl_df


FL distributed-eval accuracy: round 1 = 97.8% → round 3 = 97.8%  (+0.0% over 3 rounds)
  ↳ wrote ml\evaluation\results\fl_rounds.csv  (3 rows)
  ↳ wrote ml\evaluation\plots\fl_convergence.png


,round,fit_accuracy,fit_loss,eval_accuracy,eval_loss
0,1,0.9832,0.1546,0.9783,0.2977
1,2,0.9888,0.0925,0.9783,0.2512
2,3,0.9916,0.0469,0.9783,0.2156


## Privacy Payload Audit

The privacy claim that raw biometrics never leave the device can be evaluated as well, by enumerating every field in the payloads that cross the netword and classifying each as model-derived / biometric.
We also run a keyword scan over the codebase for raw-signal names.


In [97]:
import re

boundary_fields = [
    ("FL weight upload",       "weights_file (binary)",     "averaged model tensors",  "model-derived"),
    ("FL round summary",       "round",                     "int",                     "metadata"),
    ("FL round summary",       "num_clients",               "int",                     "metadata"),
    ("FL round summary",       "mean_loss",                 "float",                   "model-derived"),
    ("FL round summary",       "mean_accuracy",             "float",                   "model-derived"),
    ("FL round summary",       "weights_filename",          "str",                     "metadata"),
    ("HR aggregate",           "mean_stress_score",         "float (k-anon, n>=5)",    "aggregate"),
    ("HR aggregate",           "std_stress",                "float (k-anon, n>=5)",    "aggregate"),
    ("HR aggregate",           "n_users",                   "int (>=5 enforced)",      "metadata"),
    ("Journal entry",          "(NOT uploaded)",            "stays on device",         "NOT transmitted"),
    ("Feature tensor (12×23)", "(NOT uploaded)",            "stays on device",         "NOT transmitted"),
    ("Per-window prediction",  "(NOT uploaded)",            "stays on device",         "NOT transmitted"),
    ("Raw BVP signal",         "(NOT uploaded)",            "stays on device",         "NOT transmitted"),
    ("Raw EDA signal",         "(NOT uploaded)",            "stays on device",         "NOT transmitted"),
]
priv_df = pd.DataFrame(boundary_fields,
                       columns=["payload", "field", "type", "category"])
priv_df["raw_biometric"] = (priv_df["category"] == "raw_biometric").map({True: "YES", False: "no"})
n_raw_boundary = (priv_df["category"] == "raw_biometric").sum()

RAW_PATTERNS = re.compile(
    r"\b(bvp|eda|scr|scl|rmssd|sdnn|raw_signal|raw_sample|"
    r"biometric_sample|sensor_window|heart_rate_series|"
    r"journal_text|diary_entry|feature_tensor)\b", re.IGNORECASE)
server_app_dir = REPO_ROOT / "server" / "app"
hits = []
for py in server_app_dir.rglob("*.py"):
    for lineno, line in enumerate(py.read_text(encoding="utf-8").splitlines(), 1):
        code = line.split("#", 1)[0]
        if RAW_PATTERNS.search(code) and ("=" in code or ":" in code) \
           and "no " not in code.lower():
            hits.append(f"{py.relative_to(REPO_ROOT)}:{lineno}: {line.strip()}")

def _parse_server_tests(stdout: str, file_pattern: str) -> list[dict]:
    rows = []
    for line in stdout.splitlines():
        if re.search(file_pattern, line) and re.search(r"PASSED|FAILED|SKIPPED", line):
            test_id = line.strip().split(" ")[0]
            name = test_id.split("::")[-1] if "::" in test_id else test_id
            status = "PASS" if "PASSED" in line else ("SKIP" if "SKIPPED" in line else "FAIL")
            rows.append({"test": name, "status": status, "line": line.strip()})
    return rows

kanon_tests = _parse_server_tests(server_result.stdout, r"test_stats_kanon")
auth_tests  = _parse_server_tests(server_result.stdout, r"test_auth")

kanon_pass_rate = (sum(r["status"]=="PASS" for r in kanon_tests) / len(kanon_tests)
                   if kanon_tests else float("nan"))
auth_pass_rate  = (sum(r["status"]=="PASS" for r in auth_tests) / len(auth_tests)
                   if auth_tests else float("nan"))

security_rows = [
    {"check": "Raw biometric fields crossing boundary",    "count": int(n_raw_boundary), "target": 0,
     "verdict": "PASS" if n_raw_boundary==0 else "FAIL"},
    {"check": "Raw-signal keywords in server code",        "count": len(hits), "target": 0,
     "verdict": "PASS" if not hits else "REVIEW"},
    {"check": "Journal/feature-tensor/prediction uploads", "count": 0, "target": 0,
     "verdict": "PASS"},
    {"check": f"k-anon rejection tests ({len(kanon_tests)} cases, n<5 → 422)",
     "count": len(kanon_tests), "target": ">0",
     "verdict": f"{kanon_pass_rate:.0%}" if kanon_tests else "re-run §1 server tests"},
    {"check": f"Auth/role rejection tests ({len(auth_tests)} cases, 401/403)",
     "count": len(auth_tests), "target": ">0",
     "verdict": f"{auth_pass_rate:.0%}" if auth_tests else "re-run §1 server tests"},
]
sec_df = pd.DataFrame(security_rows)

print(f"Boundary fields audited       : {len(priv_df)}")
print(f"Raw-biometric fields crossing : {n_raw_boundary}")
print(f"Raw-signal server code hits   : {len(hits)}")
if hits:
    for h in hits: print("ISSUE", h)
if kanon_tests:
    print(f"k-anon tests: {len(kanon_tests)} cases  pass={kanon_pass_rate:.0%}")
    for r in kanon_tests: print(f"  {r['status']:4s}  {r['test']}")
else:
    print("k-anon: no test lines found in server_result - re-run cell #VSC-3cc8f92d first")
if auth_tests:
    print(f"Auth tests: {len(auth_tests)} cases  pass={auth_pass_rate:.0%}")
else:
    print("Auth: no test lines found - re-run cell #VSC-3cc8f92d first")
verdict_str = "PASS" if (n_raw_boundary==0 and not hits) else "REVIEW"
print(f"\nPrivacy audit: {verdict_str}")

priv_out = pd.concat([
    priv_df,
    pd.DataFrame([{"payload": "SCAN", "field": "server raw-signal field defs",
                   "type": f"{len(hits)} matches", "category": "audit",
                   "raw_biometric": "YES" if hits else "no"}]),
], ignore_index=True)
save_table(priv_out, "privacy_payload_audit.csv",
           "Network-boundary field classification + server raw-signal keyword scan")
save_table(sec_df, "security_tests.csv",
           "Privacy and security check summary: k-anon, auth/role tests, payload audit")
sec_df


Boundary fields audited       : 14
Raw-biometric fields crossing : 0
Raw-signal server code hits   : 0
k-anon tests: 7 cases  pass=100%
  PASS  test_n_users_below_threshold_rejected
  PASS  test_n_users_at_threshold_accepted
  PASS  test_n_users_above_threshold_accepted
  PASS  test_n_users_1_rejected
  PASS  test_cross_org_rejected
  PASS  test_requires_hr_or_admin_role
  PASS  test_admin_can_list
Auth tests: 13 cases  pass=100%

Privacy audit: PASS
  ↳ wrote ml\evaluation\results\privacy_payload_audit.csv  (15 rows)
  ↳ wrote ml\evaluation\results\security_tests.csv  (5 rows)


,check,count,target,verdict
0,Raw biometric fields crossing boundary,0,0,PASS
1,Raw-signal keywords in server code,0,0,PASS
2,Journal/feature-tensor/prediction uploads,0,0,PASS
3,"k-anon rejection tests (7 cases, n<5 → 422)",7,>0,100%
4,"Auth/role rejection tests (13 cases, 401/403)",13,>0,100%


## Test checker for Mobile & Wear modules

This step checks if all the unit tests associated to mobile/wear functionalities pass.

In [98]:
# Wearable integration test results: parse Gradle JUnit XML from mobile + wear modules
import xml.etree.ElementTree as ET

def parse_junit_xml(xml_dir: Path) -> list[dict]:
    rows = []
    if not xml_dir.exists():
        return rows
    for xf in sorted(xml_dir.glob("TEST-*.xml")):
        tree = ET.parse(xf)
        root = tree.getroot()
        suite_name = root.attrib.get("name", xf.stem)
        suite_time = float(root.attrib.get("time", 0))
        for tc in root.iter("testcase"):
            failure = tc.find("failure")
            skipped = tc.find("skipped")
            rows.append({
                "suite": suite_name.split(".")[-1],
                "test": tc.attrib.get("name", ""),
                "time_ms": round(float(tc.attrib.get("time", 0)) * 1000, 1),
                "status": "SKIP" if skipped is not None else
                          ("FAIL" if failure is not None else "PASS"),
            })
    return rows

mobile_xml = REPO_ROOT / "mobile" / "build" / "test-results" / "testDebugUnitTest"
wear_xml   = REPO_ROOT / "wear"   / "build" / "test-results" / "testDebugUnitTest"

mob_rows  = parse_junit_xml(mobile_xml)
wear_rows = parse_junit_xml(wear_xml)

for module, rws in [("mobile", mob_rows), ("wear", wear_rows)]:
    for r in rws:
        r["module"] = module

unit_df = pd.DataFrame(mob_rows + wear_rows)
if not unit_df.empty:
    summary = unit_df.groupby(["module","suite"]).agg(
        tests=("test","count"), passed=("status", lambda x: (x=="PASS").sum()),
        failed=("status", lambda x: (x=="FAIL").sum()),
        total_ms=("time_ms","sum")
    ).reset_index()
    pass_rate = (unit_df["status"] == "PASS").mean()
    print(f"Android unit tests: {len(unit_df)} cases  "
          f"pass={int((unit_df.status=='PASS').sum())}  "
          f"fail={int((unit_df.status=='FAIL').sum())}  "
          f"({pass_rate:.1%} pass rate)")
    print(summary.to_string(index=False))
    save_table(unit_df, "wearable_unit_tests.csv",
               "Android unit-test results parsed from Gradle JUnit XML (mobile + wear)")
    
    wear_only = unit_df[unit_df.module == "wear"]
    print(f"\nWear-only tests: {len(wear_only)}  "
          f"(DataMap schema: {(wear_only.suite.str.contains('DataMap')).sum()} cases, "
          f"Buffer flush: {(wear_only.suite.str.contains('Buffer')).sum()} cases)")
else:
    print("No Gradle test XML found - re-run `:mobile:testDebugUnitTest` and `:wear:testDebugUnitTest`")
unit_df


Android unit tests: 40 cases  pass=40  fail=0  (100.0% pass rate)
module                 suite  tests  passed  failed  total_ms
mobile  FeatureExtractorTest      8       8       0      36.0
mobile  ScalerNormalizerTest      5       5       0       9.0
mobile       XaiGroupingTest      9       9       0       9.0
  wear SensorBufferFlushTest      7       7       0      13.0
  wear WearDataMapSchemaTest     11      11       0      24.0
  ↳ wrote ml\evaluation\results\wearable_unit_tests.csv  (40 rows)

Wear-only tests: 18  (DataMap schema: 11 cases, Buffer flush: 7 cases)


,suite,test,time_ms,status,module
0,FeatureExtractorTest,extract returns correct shape for valid input,24.0,PASS,mobile
1,FeatureExtractorTest,output is deterministic,4.0,PASS,mobile
2,FeatureExtractorTest,extract returns null for very short HR,1.0,PASS,mobile
3,FeatureExtractorTest,empty ACC still produces valid output,1.0,PASS,mobile
4,FeatureExtractorTest,no NaN or Inf in output,5.0,PASS,mobile
5,FeatureExtractorTest,temp features contain correct mean for constan...,1.0,PASS,mobile
6,FeatureExtractorTest,constants match Python pipeline,0.0,PASS,mobile
7,FeatureExtractorTest,HRV features use correct RR interval formula,0.0,PASS,mobile
8,ScalerNormalizerTest,normalize produces correct z-scores,3.0,PASS,mobile
9,ScalerNormalizerTest,normalize returns input unchanged when dimensi...,5.0,PASS,mobile


In [99]:
centralised_acc = mean_acc
fl_eval_final   = eval_acc[rounds[-1]]

n_virtual_clients = 15

import tensorflow as tf
km2 = tf.keras.models.load_model(str(MODELS_DIR / "mindwave_stress.keras"))
w2 = km2.get_weights()
weight_bytes_per_client = sum(a.nbytes for a in w2)
weight_bytes_int8       = weight_bytes_per_client // 4  
del km2, w2

total_upload_3rounds   = weight_bytes_per_client * n_virtual_clients * len(rounds)
total_download_3rounds = weight_bytes_per_client * n_virtual_clients * len(rounds)

comm_rows = [
    {"direction": "Upload (client→server)", "encoding": "float32",
     "bytes_per_client_per_round": weight_bytes_per_client,
     "kb_per_client_per_round": round(weight_bytes_per_client/1024, 1),
     "total_bytes_all_rounds": total_upload_3rounds,
     "total_kb_all_rounds": round(total_upload_3rounds/1024, 1)},
    {"direction": "Upload (client→server)", "encoding": "int8 (potential)",
     "bytes_per_client_per_round": weight_bytes_int8,
     "kb_per_client_per_round": round(weight_bytes_int8/1024, 1),
     "total_bytes_all_rounds": weight_bytes_int8 * n_virtual_clients * len(rounds),
     "total_kb_all_rounds": round(weight_bytes_int8 * n_virtual_clients * len(rounds)/1024, 1)},
    {"direction": "Download (server→client)", "encoding": "float32",
     "bytes_per_client_per_round": weight_bytes_per_client,
     "kb_per_client_per_round": round(weight_bytes_per_client/1024, 1),
     "total_bytes_all_rounds": total_download_3rounds,
     "total_kb_all_rounds": round(total_download_3rounds/1024, 1)},
]
comm_df = pd.DataFrame(comm_rows)

gap_rows = [
    {"evaluation": "Centralised - LOSO (cross-subject)", "accuracy": round(centralised_acc, 4),
     "note": "upper bound - full data access at training"},
    {"evaluation": f"FL round {rounds[-1]} distributed eval ({n_virtual_clients} virtual clients)",
     "accuracy": round(fl_eval_final, 4),
     "note": f"warm-start, {len(rounds)} rounds, FedAvg"},
    {"evaluation": "Accuracy gap (centralised − FL round 3)", "accuracy": round(centralised_acc - fl_eval_final, 4),
     "note": "expected to shrink with more rounds / local epochs"},
]
gap_df = pd.DataFrame(gap_rows)

print("Centralised vs FL performance:")
print(gap_df.to_string(index=False))
print(f"\nVirtual clients per round  : {n_virtual_clients}")
print(f"FL rounds run              : {len(rounds)}")
print(f"Upload per client/round    : {weight_bytes_per_client:,} bytes ({weight_bytes_per_client/1024:.1f} KB)")
print(f"Total upload {len(rounds)} rounds      : {total_upload_3rounds/1024:.1f} KB "
      f"({total_upload_3rounds/1024/1024:.2f} MB)")

save_table(gap_df, "fl_centralised_vs_fl.csv", "Centralised vs FL accuracy gap (LOSO vs FL round 3)")
save_table(comm_df, "fl_comm_cost.csv",
           "FL communication cost per client per round (float32 and potential int8 encoding)")
gap_df


Centralised vs FL performance:
                                      evaluation  accuracy                                               note
              Centralised - LOSO (cross-subject)    0.9520         upper bound - full data access at training
FL round 3 distributed eval (15 virtual clients)    0.9783                       warm-start, 3 rounds, FedAvg
         Accuracy gap (centralised − FL round 3)   -0.0262 expected to shrink with more rounds / local epochs

Virtual clients per round  : 15
FL rounds run              : 3
Upload per client/round    : 144,264 bytes (140.9 KB)
Total upload 3 rounds      : 6339.7 KB (6.19 MB)
  ↳ wrote ml\evaluation\results\fl_centralised_vs_fl.csv  (3 rows)
  ↳ wrote ml\evaluation\results\fl_comm_cost.csv  (3 rows)


,evaluation,accuracy,note
0,Centralised - LOSO (cross-subject),0.9520,upper bound - full data access at training
1,FL round 3 distributed eval (15 virtual clients),0.9783,"warm-start, 3 rounds, FedAvg"
2,Accuracy gap (centralised − FL round 3),-0.0262,expected to shrink with more rounds / local ep...


## Backend Write Latency

This section measures the trip time from the mobile app sending a k-anon aggregate to the FastAPI backend until PostgreSQL confirms the write. This covers the entire stack, from HTTP, to FastAPI, SQLAlchemy, PostgreSQL.

This part require the containers to be up.

In [100]:
import urllib.request, urllib.error, urllib.parse, json as _json

API_BASE = "http://localhost:8000"
DB_REPS  = 30
DB_WARMUP = 5

def _api_health() -> bool:
    try:
        with urllib.request.urlopen(f"{API_BASE}/health", timeout=3) as r:
            return r.status == 200
    except Exception:
        return False

def _get_token(email: str, password: str):
    """POST to /auth/login (OAuth2PasswordRequestForm - form-encoded)."""
    data = urllib.parse.urlencode({"username": email, "password": password}).encode()
    req  = urllib.request.Request(f"{API_BASE}/auth/login", data=data, method="POST",
                                   headers={"Content-Type": "application/x-www-form-urlencoded"})
    try:
        with urllib.request.urlopen(req, timeout=5) as r:
            return _json.loads(r.read())["access_token"]
    except Exception as exc:
        print(f"  Login failed: {exc}")
        return None

def _post_json(url: str, body: dict, token: str) -> int:
    data = _json.dumps(body).encode()
    req  = urllib.request.Request(url, data=data, method="POST",
                                   headers={"Content-Type": "application/json",
                                            "Authorization": f"Bearer {token}"})
    try:
        with urllib.request.urlopen(req, timeout=5) as r:
            return r.status
    except urllib.error.HTTPError as e:
        return e.code

if not _api_health():
    print("API not reachable - skipping DB latency. Start with: docker compose up -d")
    db_lat_df = pd.DataFrame([{"operation": "DB write (API not running)",
                                "reps": 0, "mean_ms": float("nan"),
                                "p50_ms": float("nan"), "p95_ms": float("nan"),
                                "p99_ms": float("nan"), "max_ms": float("nan")}])
else:
    from datetime import datetime, timezone as _tz, timedelta as _td
    token = _get_token("admin@demo.mindwave.app", "Admin1234!")
    if not token:
        print("  Seed first: docker compose exec api python -m seed_demo")
        db_lat_df = pd.DataFrame([{"operation": "DB write (not authenticated)",
                                    "reps": 0, "mean_ms": float("nan"),
                                    "p50_ms": float("nan"), "p95_ms": float("nan"),
                                    "p99_ms": float("nan"), "max_ms": float("nan")}])
    else:
        _base = datetime(2025, 8, 1, tzinfo=_tz.utc)
        db_times = []
        for i in range(DB_REPS + DB_WARMUP):
            ts_start = _base + _td(hours=i)
            ts_end   = ts_start + _td(minutes=30)
            body = {
                "organization_id": 1,
                "period_start": ts_start.isoformat(),
                "period_end": ts_end.isoformat(),
                "period_type": "custom",
                "mean_stress_score": 0.50 + (i % 5) * 0.05,
                "std_stress": 0.1,
                "n_users": 10 + i % 40,
            }
            t0 = time.perf_counter()
            sc = _post_json(f"{API_BASE}/stats/aggregate", body, token)
            dt = (time.perf_counter() - t0) * 1000.0
            if i >= DB_WARMUP:
                db_times.append(dt)

        db_t = np.asarray(db_times)
        db_lat_df = pd.DataFrame([{
            "operation": "DB write (/stats/aggregate via FastAPI+SQLAlchemy+PostgreSQL)",
            "reps": len(db_t), "mean_ms": round(db_t.mean(), 1),
            "p50_ms": round(np.percentile(db_t, 50), 1),
            "p95_ms": round(np.percentile(db_t, 95), 1),
            "p99_ms": round(np.percentile(db_t, 99), 1),
            "max_ms": round(db_t.max(), 1),
        }])
        print(f"DB write latency: mean={db_t.mean():.1f} ms  "
              f"p50={np.percentile(db_t,50):.1f} ms  p95={np.percentile(db_t,95):.1f} ms")

save_table(db_lat_df, "db_write_latency.csv",
           "HTTP→FastAPI→SQLAlchemy→PostgreSQL write latency per request (Docker Compose local)")
db_lat_df


DB write latency: mean=12.3 ms  p50=11.8 ms  p95=14.8 ms
  ↳ wrote ml\evaluation\results\db_write_latency.csv  (1 rows)


,operation,reps,mean_ms,p50_ms,p95_ms,p99_ms,max_ms
0,DB write (/stats/aggregate via FastAPI+SQLAlch...,30,12.3,11.8,14.8,16.6,17.3
